# Demo 1: DataOps és automatizáció

**Kapcsolódó diák:** 4–13 (DataOps fogalma, CI/CD, tesztelési piramis, observability)

**Előfeltétel:** `docker compose up -d` – JupyterLab: http://localhost:8888 (token: demo)

Tartalom:
1. Unit tesztek – transzformációs logika ellenőrzése
2. Pipeline observability – SLA monitor, sor szám, freshness check
3. Data contract validálás
4. CI/CD integrációs példa


In [1]:
!pip install -q pandas pytest

## 1. Unit tesztek – transzformációs logika

A tesztelési piramis alján a unit tesztek állnak: gyorsak, izoláltak, és egy-egy
transzformációs függvény logikáját ellenőrzik.


In [2]:
import pandas as pd

# ── A tesztelendő transzformációs függvények ────────────────────────────
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    """Silver réteg transzformáció: NULL és negatív érték szűrés."""
    return (
        df
        .dropna(subset=['order_id', 'customer_id'])
        .query('amount > 0')
        .query('status == "shipped"')
    )

def compute_daily_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Gold réteg: napi bevétel aggregálás csak shipped rendelésekből."""
    return (
        df
        .groupby('order_date')
        .agg(revenue=('amount', 'sum'), order_count=('order_id', 'count'))
        .reset_index()
    )

print('Transzformációs függvények definiálva.')

Transzformációs függvények definiálva.


In [3]:
# ── Unit tesztek – pytest nélkül, közvetlenül a notebookban futtatva ───
import pandas as pd

sample_orders = pd.DataFrame({
    'order_id':    [1, 2, 3, None],
    'customer_id': [101, 102, None, 104],
    'amount':      [12500.0, -100.0, 8750.0, 450.0],
    'order_date':  pd.to_datetime(['2024-01-15', '2024-01-15', '2024-01-16', '2024-01-16']),
    'status':      ['shipped', 'cancelled', 'shipped', 'pending'],
})

# Test 1: NULL order_id kiszűrése
result = clean_orders(sample_orders)
assert result['order_id'].notna().all(), 'NULL order_id nem szűrődött ki!'
print('✅ Test 1 PASS: NULL order_id kiszűrve')

# Test 2: negatív amount kiszűrése
assert (result['amount'] > 0).all(), 'Negatív amount maradt!'
print('✅ Test 2 PASS: negatív amount kiszűrve')

# Test 3: napi bevétel – csak shipped státusz
revenue = compute_daily_revenue(result)
jan15 = revenue[revenue['order_date'] == '2024-01-15']['revenue'].values[0]
assert jan15 == 12500.0, f'Várt 12500.0, kapott {jan15}'
print(f'✅ Test 3 PASS: jan.15 bevétel = {jan15:,.0f} Ft')

print(f'\n{len(result)} sor maradt (volt {len(sample_orders)})')

✅ Test 1 PASS: NULL order_id kiszűrve
✅ Test 2 PASS: negatív amount kiszűrve
✅ Test 3 PASS: jan.15 bevétel = 12,500 Ft

1 sor maradt (volt 4)


## 2. Pipeline observability – SLA monitor

Az observability a pipeline állapotának folyamatos megfigyelése: futási idő,
sor szám és adatfrissesség ellenőrzése.


In [4]:
import time
from datetime import datetime, timedelta
from typing import Optional
import pandas as pd

class PipelineMonitor:
    """Egyszerű pipeline SLA és freshness monitor."""

    def __init__(self, pipeline_name: str, sla_minutes: int = 60):
        self.pipeline_name = pipeline_name
        self.sla_minutes   = sla_minutes
        self._start_time: Optional[float] = None
        self.metrics: dict = {}

    def start(self):
        self._start_time = time.time()
        print(f'[{self.pipeline_name}] ▶ Indítás: {datetime.now().strftime("%H:%M:%S")}')

    def stop(self) -> float:
        elapsed = time.time() - self._start_time
        self.metrics['duration_sec'] = round(elapsed, 2)
        status = '✅' if elapsed <= self.sla_minutes * 60 else '⚠️ SLA BREACH'
        print(f'[{self.pipeline_name}] ■ Futási idő: {elapsed:.1f}s {status}')
        return elapsed

    def check_row_count(self, df: pd.DataFrame, min_rows: int = 1,
                        max_rows: int = 10_000_000) -> bool:
        n = len(df)
        self.metrics['row_count'] = n
        ok = min_rows <= n <= max_rows
        icon = '✅' if ok else '❌'
        print(f'[{self.pipeline_name}] {icon} Sorok: {n:,} (elvárt: {min_rows:,}–{max_rows:,})')
        if not ok:
            raise RuntimeError(f'Sor szám ({n:,}) kívül esik a várt tartományon!')
        return ok

    def check_freshness(self, df: pd.DataFrame, date_col: str,
                        max_age_hours: int = 25) -> bool:
        latest    = pd.to_datetime(df[date_col]).max()
        age_hours = (datetime.now() - latest.to_pydatetime()).total_seconds() / 3600
        self.metrics['data_age_hours'] = round(age_hours, 1)
        ok   = age_hours <= max_age_hours
        icon = '✅' if ok else '❌'
        print(f'[{self.pipeline_name}] {icon} Adat kora: {age_hours:.1f}h (max: {max_age_hours}h)')
        if not ok:
            raise RuntimeError(f'Adat túl régi: {age_hours:.1f}h > {max_age_hours}h SLA!')
        return ok

    def summary(self):
        print(f'\n[{self.pipeline_name}] 📊 Összefoglaló: {self.metrics}')


# Futtatás
df = pd.DataFrame({
    'order_id':   range(1, 4873),
    'order_date': [datetime.now() - timedelta(hours=2)] * 4872,
    'amount':     [1000.0] * 4872,
})

monitor = PipelineMonitor('orders_silver_pipeline', sla_minutes=30)
monitor.start()
time.sleep(0.1)
monitor.stop()
monitor.check_row_count(df, min_rows=1000)
monitor.check_freshness(df, date_col='order_date', max_age_hours=25)
monitor.summary()

[orders_silver_pipeline] ▶ Indítás: 07:58:55
[orders_silver_pipeline] ■ Futási idő: 0.1s ✅
[orders_silver_pipeline] ✅ Sorok: 4,872 (elvárt: 1,000–10,000,000)
[orders_silver_pipeline] ✅ Adat kora: 2.0h (max: 25h)

[orders_silver_pipeline] 📊 Összefoglaló: {'duration_sec': 0.1, 'row_count': 4872, 'data_age_hours': 2.0}


## 3. Data contract validálás

A data contract az upstream csapat ígérete a downstream csapatnak:
definiálja a séma elvárásokat, NULL-szabályokat és értékkészleteket.


In [5]:
import pandas as pd

ORDERS_CONTRACT = {
    'dataset':  'orders',
    'version':  '1.2.0',
    'owner':    'sales-domain-team',
    'sla': {'freshness_hours': 24, 'min_rows': 1000},
    'schema': {
        'order_id':    {'type': 'int64',   'nullable': False, 'unique': True},
        'customer_id': {'type': 'int64',   'nullable': False},
        'amount':      {'type': 'float64', 'nullable': False, 'min': 0.01},
        'status':      {'type': 'object',  'nullable': False,
                        'allowed_values': ['shipped', 'pending', 'cancelled', 'returned']},
    }
}

def validate_contract(df: pd.DataFrame, contract: dict) -> bool:
    errors = []
    for col, rules in contract['schema'].items():
        if col not in df.columns:
            errors.append(f'Hiányzó oszlop: {col}'); continue
        if not rules['nullable'] and df[col].isna().any():
            errors.append(f'NULL értékek: {col}')
        if rules.get('unique') and df[col].duplicated().any():
            errors.append(f'Duplikált értékek: {col}')
        if 'min' in rules and (df[col] < rules['min']).any():
            errors.append(f'{col} < {rules["min"]}')
        if 'allowed_values' in rules:
            invalid = ~df[col].isin(rules['allowed_values'])
            if invalid.any():
                errors.append(f'Érvénytelen értékek {col}: {df.loc[invalid, col].unique()}')
    if errors:
        print('❌ Data contract VIOLATED:')
        for e in errors: print(f'   - {e}')
        return False
    print(f'✅ Data contract OK: {contract["dataset"]} v{contract["version"]}')
    return True


# Helyes adat
df_ok = pd.DataFrame({
    'order_id': [1, 2, 3], 'customer_id': [101, 102, 103],
    'amount': [12500.0, 3200.0, 8750.0],
    'status': ['shipped', 'pending', 'cancelled'],
})
validate_contract(df_ok, ORDERS_CONTRACT)

# Hibás adat
df_bad = pd.DataFrame({
    'order_id': [1, 1, 3], 'customer_id': [101, None, 103],
    'amount': [12500.0, -50.0, 8750.0],
    'status': ['shipped', 'SHIPPED', 'unknown'],
})
validate_contract(df_bad, ORDERS_CONTRACT)

✅ Data contract OK: orders v1.2.0
❌ Data contract VIOLATED:
   - Duplikált értékek: order_id
   - NULL értékek: customer_id
   - amount < 0.01
   - Érvénytelen értékek status: ['SHIPPED' 'unknown']


False

## 4. CI/CD pipeline szimulálása

Egy teljes DataOps CI/CD ciklus: lint → unit test → DQ gate → deploy döntés.


In [6]:
import pandas as pd

def simulate_ci_pipeline(df: pd.DataFrame) -> bool:
    """Szimulált DataOps CI/CD lépések."""
    print('=== DataOps CI Pipeline ===\n')

    # Step 1: Unit tesztek
    print('Step 1: Unit tesztek...')
    try:
        cleaned = clean_orders(df)
        assert len(cleaned) > 0
        print('  ✅ Unit tesztek PASS')
    except Exception as e:
        print(f'  ❌ Unit tesztek FAIL: {e}'); return False

    # Step 2: Data contract
    print('Step 2: Data contract validálás...')
    cols_ok = all(c in df.columns for c in ['order_id', 'customer_id', 'amount', 'status'])
    if not cols_ok:
        print('  ❌ Hiányzó oszlop!'); return False
    print('  ✅ Contract OK')

    # Step 3: DQ gate
    print('Step 3: DQ gate...')
    null_rate = df['customer_id'].isna().mean()
    if null_rate > 0.05:
        print(f'  ❌ DQ gate FAIL: customer_id NULL arány {null_rate:.0%} > 5%'); return False
    print(f'  ✅ DQ gate PASS: NULL arány {null_rate:.0%}')

    print('\n🚀 Deploy: ENGEDÉLYEZETT → Gold réteg feltöltés')
    return True


df_pipeline = pd.DataFrame({
    'order_id': range(1, 101),
    'customer_id': [i if i % 20 != 0 else None for i in range(1, 101)],
    'amount': [float(i * 100) for i in range(1, 101)],
    'status': ['shipped'] * 100,
})

simulate_ci_pipeline(df_pipeline)

=== DataOps CI Pipeline ===

Step 1: Unit tesztek...
  ✅ Unit tesztek PASS
Step 2: Data contract validálás...
  ✅ Contract OK
Step 3: DQ gate...
  ✅ DQ gate PASS: NULL arány 5%

🚀 Deploy: ENGEDÉLYEZETT → Gold réteg feltöltés


True